# Intelligent Library Chat Assistant - Step-by-Step Demo

## Architecture & Evaluation

This notebook demonstrates the **inner workings** of our Intelligent Library Chat Assistant system. 
Following the architecture from Chapter 3, we show each component's input, processing, and output.

### System Architecture Overview:
1. **Input Processing** -> Text preprocessing & tokenization
2. **Feature Extraction** -> Bag-of-Words & TF-IDF
3. **Intent Classification** -> ML-based classification
4. **Rule Engine** -> Pattern matching for specific queries
5. **Hybrid Integration** -> Combining ML + Rules
6. **Response Generation** -> Template-based responses

Let's begin!


## Cell 1: Environment Setup & Library Versions

In [1]:
# Import required libraries
import sys
import json
import re
from collections import Counter

# NLP Libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML Libraries
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn import __version__ as sklearn_version

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Display versions
print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)
print(f"Python Version: {sys.version}")
print(f"NLTK Version: {nltk.__version__}")
print(f"Sklearn Version: {sklearn_version}")
print("\nAll libraries loaded successfully!")


ENVIRONMENT SETUP
Python Version: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
NLTK Version: 3.9.3
Sklearn Version: 1.7.2

All libraries loaded successfully!


## Cell 2: Load Training Data from intent_examples.json

In [2]:
# Load training data from intent_examples.json (same as main project)
import json

# Load intent examples from JSON file
with open('app/data/intent_examples.json', 'r', encoding='utf-8') as f:
    intent_data = json.load(f)

# Convert to training data format (text, label) pairs
training_data = []
for intent, examples in intent_data.items():
    for example in examples:
        training_data.append((example, intent))

# Separate into texts and labels
texts, labels = zip(*training_data)

print("=" * 60)
print("TRAINING DATA LOADED FROM intent_examples.json")
print("=" * 60)
print(f"Total training samples: {len(texts)}")
print("\nIntent distribution:")
for intent, count in Counter(labels).items():
    print(f"  - {intent}: {count} samples")

print("\nSample data points:")
for i in range(3):
    print(f'  Text: "{texts[i]}"')
    print(f"  Intent: {labels[i]}")
    print()


TRAINING DATA LOADED FROM intent_examples.json
Total training samples: 135

Intent distribution:
  - book_search: 16 samples
  - borrow_request: 13 samples
  - borrowing_info: 10 samples
  - return: 10 samples
  - reservation: 10 samples
  - library_hours: 8 samples
  - policy_query: 8 samples
  - research_help: 8 samples
  - citation_help: 8 samples
  - general_info: 8 samples
  - introduction: 5 samples
  - about_you: 5 samples
  - bot_identity: 5 samples
  - bot_purpose: 5 samples
  - my_borrows: 8 samples
  - my_reservations: 8 samples

Sample data points:
  Text: "Find books about computer science"
  Intent: book_search

  Text: "search for books"
  Intent: book_search

  Text: "how do i search for books"
  Intent: book_search



## Cell 3: Text Preprocessing - Tokenization

In [3]:
# Text Preprocessing Functions
def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # 3. Tokenize
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    stop_words = set(stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop_words]
    # 5. Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# Demonstrate preprocessing
print("=" * 60)
print("TEXT PREPROCESSING DEMO")
print("=" * 60)

sample_texts = [
    "I want to find a book about Python programming",
    "Are you open on weekends?",
    "Can I borrow Clean Code today?",
    "Who are you?"
]

for text in sample_texts:
    tokens = preprocess_text(text)
    print(f'\nOriginal: "{text}"')
    print(f"Tokens: {tokens}")
    print(f"Token count: {len(tokens)}")


TEXT PREPROCESSING DEMO

Original: "I want to find a book about Python programming"
Tokens: ['want', 'find', 'book', 'python', 'programming']
Token count: 5

Original: "Are you open on weekends?"
Tokens: ['open', 'weekend']
Token count: 2

Original: "Can I borrow Clean Code today?"
Tokens: ['borrow', 'clean', 'code', 'today']
Token count: 4

Original: "Who are you?"
Tokens: []
Token count: 0


## Cell 4: Feature Extraction - Bag of Words (BoW)

In [4]:
# Apply preprocessing to all training data
processed_texts = [" ".join(preprocess_text(t)) for t in texts]

# Create Bag-of-Words vectorizer
bow_vectorizer = CountVectorizer(max_features=100, min_df=1)
bow_matrix = bow_vectorizer.fit_transform(processed_texts)

print("=" * 60)
print("BAG-OF-WORDS FEATURE EXTRACTION")
print("=" * 60)

print(f"\nFeature Matrix Shape: {bow_matrix.shape}")
print(f"   - Rows (documents): {bow_matrix.shape[0]}")
print(f"   - Columns (vocabulary size): {bow_matrix.shape[1]}")

# Show vocabulary
vocabulary = bow_vectorizer.get_feature_names_out()
print(f"\nVocabulary (first 20 words):")
print(f"   {list(vocabulary[:20])}")

# Show a sample BoW vector
print(f"\nSample BoW vector (first document):")
sample_vector = bow_matrix[0].toarray()[0]
non_zero = [(vocabulary[i], sample_vector[i]) for i in range(len(sample_vector)) if sample_vector[i] > 0]
print(f"   {dict(non_zero)}")


BAG-OF-WORDS FEATURE EXTRACTION

Feature Matrix Shape: (135, 100)
   - Rows (documents): 135
   - Columns (vocabulary size): 100

Vocabulary (first 20 words):
   ['ai', 'allowed', 'apa', 'article', 'artificial', 'author', 'available', 'book', 'borrow', 'borrowed', 'borrowing', 'borrows', 'bot', 'built', 'called', 'capability', 'check', 'citation', 'cite', 'closing']

Sample BoW vector (first document):
   {'book': np.int64(1), 'computer': np.int64(1), 'find': np.int64(1), 'science': np.int64(1)}


## Cell 5: Feature Extraction - TF-IDF

In [5]:
# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100, min_df=1)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_texts)

print("=" * 60)
print("TF-IDF FEATURE EXTRACTION")
print("=" * 60)

print(f"\nTF-IDF Matrix Shape: {tfidf_matrix.shape}")

# Compare BoW vs TF-IDF
test_word = "book"
if test_word in vocabulary:
    idx = list(vocabulary).index(test_word)
    print(f"\nComparison for '{test_word}':")
    print(f"   BoW:   {bow_matrix[0, idx]}")
    print(f"   TF-IDF: {tfidf_matrix[0, idx]:.4f}")

# Show TF-IDF scores for first document
print(f"\nTF-IDF scores (first document, top features):")
sample_tfidf = tfidf_matrix[0].toarray()[0]
top_indices = sample_tfidf.argsort()[-5:][::-1]
for idx in top_indices:
    if sample_tfidf[idx] > 0:
        print(f"   {vocabulary[idx]}: {sample_tfidf[idx]:.4f}")


TF-IDF FEATURE EXTRACTION

TF-IDF Matrix Shape: (135, 100)

Comparison for 'book':
   BoW:   1
   TF-IDF: 0.2384

TF-IDF scores (first document, top features):
   computer: 0.6282
   science: 0.5794
   find: 0.4613
   book: 0.2384


## Cell 6: Model Training - Intent Classification (Train on All Data)

In [6]:
# Train on all data (no train/test split, like the main project)
# This ensures the model learns from all available training examples
X_train = tfidf_matrix
y_train = labels

# Train Naive Bayes classifier on all data
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

# Make predictions on training data to verify learning
y_pred = classifier.predict(X_train)

print("=" * 60)
print("MODEL TRAINING - INTENT CLASSIFICATION")
print("=" * 60)

print(f"\nTraining Setup:")
print(f"   Training samples: {X_train.shape[0]} (100% of data from intent_examples.json)")
print(f"   No train/test split - training on all available data")

print(f"\nModel: Multinomial Naive Bayes")
print(f"   Classes learned: {len(classifier.classes_)} intents from intent_examples.json")

# Show predictions vs actual on training data
print(f"\nSample Predictions on Training Data:")
print("-" * 60)
for i in range(min(6, len(y_train))):
    status = "OK" if y_pred[i] == y_train[i] else "FAIL"
    print(f'[{status}] Input: "{texts[i]}"')
    print(f"   Predicted: {y_pred[i]} | Actual: {y_train[i]}")
    print()


MODEL TRAINING - INTENT CLASSIFICATION

Training Setup:
   Training samples: 135 (100% of data from intent_examples.json)
   No train/test split - training on all available data

Model: Multinomial Naive Bayes
   Classes learned: 16 intents from intent_examples.json

Sample Predictions on Training Data:
------------------------------------------------------------
[OK] Input: "Find books about computer science"
   Predicted: book_search | Actual: book_search

[OK] Input: "search for books"
   Predicted: book_search | Actual: book_search

[OK] Input: "how do i search for books"
   Predicted: book_search | Actual: book_search

[OK] Input: "how can i find books"
   Predicted: book_search | Actual: book_search

[OK] Input: "how to search for books"
   Predicted: book_search | Actual: book_search

[OK] Input: "search books"
   Predicted: book_search | Actual: book_search



## Cell 7: Model Evaluation - Cross-Validation

In [7]:
from sklearn.model_selection import cross_val_score

# Use cross-validation for evaluation (since we train on all data)
# This is consistent with the main project approach
cv_scores = cross_val_score(classifier, X_train, y_train, cv=5)

# Calculate F1-score using cross-validation (weighted for multi-class)
cv_f1_scores = cross_val_score(classifier, X_train, y_train, cv=5, scoring='f1_weighted')

# Calculate training accuracy (should be very high)
train_accuracy = (y_pred == y_train).mean()

print("=" * 60)
print("MODEL EVALUATION - CROSS-VALIDATION METRICS")
print("=" * 60)

print(f"\nUsing 5-Fold Cross-Validation (since we train on all data):")
print(f"\nOVERALL METRICS (Cross-Validated):")
print(f"   Mean CV Accuracy: {cv_scores.mean():.2%}")
print(f"   Std CV Accuracy:  {cv_scores.std():.2%}")
print(f"   CV Scores:        {[f'{s:.2%}' for s in cv_scores]}")
print()
print(f"   Mean CV F1-Score: {cv_f1_scores.mean():.2%}")
print(f"   Std CV F1-Score:  {cv_f1_scores.std():.2%}")
print(f"   CV F1 Scores:     {[f'{s:.2%}' for s in cv_f1_scores]}")

print(f"\nTraining Accuracy (on full dataset): {train_accuracy:.2%}")

print(f"\nClasses Learned ({len(classifier.classes_)} intents from intent_examples.json):")
print("-" * 60)
classes_str = ", ".join(list(classifier.classes_)[:8]) + "\n   " + ", ".join(list(classifier.classes_)[8:])
print(f"   {classes_str}")

print(f"\nNote: Model is trained on all {len(y_train)} examples from intent_examples.json")
print(f"      (same approach as the main project)")


MODEL EVALUATION - CROSS-VALIDATION METRICS

Using 5-Fold Cross-Validation (since we train on all data):

OVERALL METRICS (Cross-Validated):
   Mean CV Accuracy: 43.70%
   Std CV Accuracy:  4.91%
   CV Scores:        ['40.74%', '51.85%', '44.44%', '37.04%', '44.44%']

   Mean CV F1-Score: 34.91%
   Std CV F1-Score:  4.35%
   CV F1 Scores:     ['29.91%', '41.13%', '32.53%', '31.98%', '39.01%']

Training Accuracy (on full dataset): 71.11%

Classes Learned (16 intents from intent_examples.json):
------------------------------------------------------------
   about_you, book_search, borrow_request, borrowing_info, bot_identity, bot_purpose, citation_help, general_info
   introduction, library_hours, my_borrows, my_reservations, policy_query, research_help, reservation, return

Note: Model is trained on all 135 examples from intent_examples.json
      (same approach as the main project)


C:\Users\hp\Downloads\library-AI-chatbot v2.0\library-AI-chatbot v2.0\.venv\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
C:\Users\hp\Downloads\library-AI-chatbot v2.0\library-AI-chatbot v2.0\.venv\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
C:\Users\hp\Downloads\library-AI-chatbot v2.0\library-AI-chatbot v2.0\.venv\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  typ

## Cell 8: Rule-Based Matching Demo

In [8]:
# Rule-based pattern matching
class RuleEngine:
    def __init__(self):
        self.rules = {
            r"(?:search|find|look for)\s+(.+?)(?:book|novel)": "book_search",
            r"(?:book|novel)\s+by\s+([A-Za-z\s]+)": "book_search",
            r"(?:is\s+)?(?:the\s+)?(?:book\s+)?(.+?)\s+(?:available|in\s+stock|on\s+shelf)": "book_search",
            r"(?:what\s+)?(?:are\s+)?(?:your\s+)?(?:opening|closing)\s+hours": "library_hours",
            r"(?:forgot|lost|reset)\s+(?:my\s+)?password": "general_info",
            r"(?:how\s+do\s+I\s+)?return\s+(?:the\s+)?book": "return",
            r"(?:who\s+(?:are\s+)?you|what\s+is\s+your\s+name|tell\s+me\s+about\s+yourself)": "introduction",
            r"(?:how\s+do\s+I\s+)?borrow|checkout|reserve|hold": "borrow_request",
        }
    
    def match(self, query):
        query_lower = query.lower()
        for pattern, intent in self.rules.items():
            if re.search(pattern, query_lower):
                return intent, pattern
        return None, None

rule_engine = RuleEngine()

print("=" * 60)
print("RULE-BASED MATCHING DEMO")
print("=" * 60)

test_queries = [
    "Find me a book about Python",
    "Is Harry Potter available?",
    "What are your opening hours?",
    "I forgot my password",
    "How do I return a book?",
    "Who are you?",
    "Can I borrow this book?",
]

print("\nRule Matching Results:")
print("-" * 60)
for query in test_queries:
    intent, pattern = rule_engine.match(query)
    status = "MATCH" if intent else "NO MATCH"
    print(f'\n[{status}] Query: "{query}"')
    if intent:
        print(f"   Intent: {intent}")
        print(f"   Pattern: {pattern}")
    else:
        print(f"   Will use ML classifier")


RULE-BASED MATCHING DEMO

Rule Matching Results:
------------------------------------------------------------

[MATCH] Query: "Find me a book about Python"
   Intent: book_search
   Pattern: (?:search|find|look for)\s+(.+?)(?:book|novel)

[MATCH] Query: "Is Harry Potter available?"
   Intent: book_search
   Pattern: (?:is\s+)?(?:the\s+)?(?:book\s+)?(.+?)\s+(?:available|in\s+stock|on\s+shelf)

[MATCH] Query: "What are your opening hours?"
   Intent: library_hours
   Pattern: (?:what\s+)?(?:are\s+)?(?:your\s+)?(?:opening|closing)\s+hours

[MATCH] Query: "I forgot my password"
   Intent: general_info
   Pattern: (?:forgot|lost|reset)\s+(?:my\s+)?password

[NO MATCH] Query: "How do I return a book?"
   Will use ML classifier

[MATCH] Query: "Who are you?"
   Intent: introduction
   Pattern: (?:who\s+(?:are\s+)?you|what\s+is\s+your\s+name|tell\s+me\s+about\s+yourself)

[MATCH] Query: "Can I borrow this book?"
   Intent: borrow_request
   Pattern: (?:how\s+do\s+I\s+)?borrow|checkout|reserve|

## Cell 9: Hybrid NLP + Rules Integration

In [9]:
class HybridIntentClassifier:
    def __init__(self, ml_classifier, vectorizer, rule_engine):
        self.classifier = ml_classifier
        self.vectorizer = vectorizer
        self.rule_engine = rule_engine
    
    def classify(self, query):
        # Step 1: Try rule-based matching first
        rule_intent, pattern = self.rule_engine.match(query)
        
        if rule_intent:
            return {
                "intent": rule_intent,
                "source": "rules",
                "confidence": 1.0,
                "pattern": pattern
            }
        
        # Step 2: Use ML classifier
        processed = " ".join(preprocess_text(query))
        features = self.vectorizer.transform([processed])
        
        intent = self.classifier.predict(features)[0]
        probabilities = self.classifier.predict_proba(features)[0]
        confidence = max(probabilities)
        
        return {
            "intent": intent,
            "source": "ml",
            "confidence": confidence
        }

hybrid_classifier = HybridIntentClassifier(classifier, tfidf_vectorizer, rule_engine)

print("=" * 60)
print("HYBRID CLASSIFIER (ML + RULES)")
print("=" * 60)

test_queries = [
    "Find me a book about Python",
    "Hello there!",
    "What time do you close?",
    "Tell me about databases",
    "I want to borrow 1984"
]

print("\nHybrid Classification Results:")
print("-" * 60)
for query in test_queries:
    result = hybrid_classifier.classify(query)
    print(f'\nQuery: "{query}"')
    print(f"   Intent: {result['intent']}")
    print(f"   Source: {result['source']}")
    print(f"   Confidence: {result['confidence']:.2%}")


HYBRID CLASSIFIER (ML + RULES)

Hybrid Classification Results:
------------------------------------------------------------

Query: "Find me a book about Python"
   Intent: book_search
   Source: rules
   Confidence: 100.00%

Query: "Hello there!"
   Intent: book_search
   Source: ml
   Confidence: 11.85%

Query: "What time do you close?"
   Intent: library_hours
   Source: ml
   Confidence: 13.07%

Query: "Tell me about databases"
   Intent: research_help
   Source: ml
   Confidence: 11.10%

Query: "I want to borrow 1984"
   Intent: borrow_request
   Source: rules
   Confidence: 100.00%


## Cell 10: Response Generation (from response_templates.json)

In [10]:
# Load response templates from response_templates.json (same as main project)
import random

# Load response templates from JSON file
with open('app/data/response_templates.json', 'r', encoding='utf-8') as f:
    response_templates_data = json.load(f)

# Convert to simple format for demo (extract 'main' response for each intent)
response_templates = {}
for intent, templates in response_templates_data.items():
    if isinstance(templates, dict):
        response_templates[intent] = templates.get('main', 'I can help you with that.')
    else:
        response_templates[intent] = templates[0] if templates else 'I can help you with that.'

def generate_response(intent):
    template = response_templates.get(intent, "I am not sure how to help with that.")
    # Simple template formatting for demo - handle missing placeholders gracefully
    if '{' in template:
        try:
            return template.format(
                count=3, book_list="Sample book list", query="your query",
                current_time="10:00 AM", topic="this topic",
                topic1="Option 1", topic2="Option 2",
                information="See library for details",
                title="Book Title", author="Author Name"
            )
        except KeyError:
            return template
    return template

print("=" * 60)
print("RESPONSE GENERATION (from response_templates.json)")
print("=" * 60)

print(f"\nLoaded {len(response_templates)} response templates\n")
print("Sample Responses by Intent:")
print("-" * 60)
for intent in list(response_templates.keys())[:8]:
    response = generate_response(intent)
    # Truncate long responses for display
    display_response = response[:100] + "..." if len(response) > 100 else response
    print(f"\nIntent: {intent}")
    print(f"   Response: {display_response}")


RESPONSE GENERATION (from response_templates.json)

Loaded 22 response templates

Sample Responses by Intent:
------------------------------------------------------------

Intent: book_search
   Response: I found 3 book(s) matching your search:

Sample book list

Reply with the number (e.g., '1') to get ...

Intent: library_hours
   Response: 🕐 **Library Hours**

**Weekdays (Mon-Fri):** 8:00 AM - 10:00 PM
**Weekends (Sat-Sun):** 10:00 AM - 8...

Intent: borrowing_info
   Response: 📚 **Borrowing Policy**

**Students:**
• Maximum books: 5
• Loan period: 14 days
• Renewals: 1 renewa...

Intent: policy_query
   Response: Here's our library policy regarding {topic}:

{policy_details}

For the complete policy document, vi...

Intent: research_help
   Response: I can help you with research! Here are some resources:

**Databases Available:**
• JSTOR (Arts & Sci...

Intent: citation_help
   Response: **Citation Styles Support**

The library provides help with:
• APA 7th Edition
• MLA 9th Editio

## Cell 11: Complete End-to-End Pipeline Demo

In [11]:
def process_query(query):
    print(f"\n{'='*60}")
    print(f"USER QUERY: '{query}'")
    print(f"{'='*60}")
    
    # Step 1: Preprocessing
    print(f"\nSTEP 1: Preprocessing")
    tokens = preprocess_text(query)
    processed = " ".join(tokens)
    print(f"   Tokens: {tokens}")
    print(f'   Processed: "{processed}"')
    
    # Step 2: Classification
    print(f"\nSTEP 2: Intent Classification")
    result = hybrid_classifier.classify(query)
    print(f"   Detected Intent: {result['intent']}")
    print(f"   Classification Source: {result['source']}")
    print(f"   Confidence: {result['confidence']:.2%}")
    
    # Step 3: Response Generation
    print(f"\nSTEP 3: Response Generation")
    response = generate_response(result["intent"])
    print(f"   Response: {response}")
    
    print(f"\nFINAL OUTPUT")
    print(f"   {response}")
    
    return result

print("=" * 60)
print("COMPLETE END-TO-END PIPELINE DEMO")
print("=" * 60)

demo_queries = [
    "I want to find books about machine learning",
    "What are your opening hours?",
    "Who are you?",
    "Can I borrow The Great Gatsby?",
    "I forgot my password"
]

for query in demo_queries:
    process_query(query)


COMPLETE END-TO-END PIPELINE DEMO

USER QUERY: 'I want to find books about machine learning'

STEP 1: Preprocessing
   Tokens: ['want', 'find', 'book', 'machine', 'learning']
   Processed: "want find book machine learning"

STEP 2: Intent Classification
   Detected Intent: book_search
   Classification Source: ml
   Confidence: 27.91%

STEP 3: Response Generation
   Response: I found 3 book(s) matching your search:

Sample book list

Reply with the number (e.g., '1') to get more details about a specific book, or ask me to reserve one.

FINAL OUTPUT
   I found 3 book(s) matching your search:

Sample book list

Reply with the number (e.g., '1') to get more details about a specific book, or ask me to reserve one.

USER QUERY: 'What are your opening hours?'

STEP 1: Preprocessing
   Tokens: ['opening', 'hour']
   Processed: "opening hour"

STEP 2: Intent Classification
   Detected Intent: library_hours
   Classification Source: rules
   Confidence: 100.00%

STEP 3: Response Generation
   R

## Cell 12: Summary & Key Takeaways

This notebook demonstrated the **inner workings** of our Intelligent Library Chat Assistant.

### What We Covered:
- **Environment Setup** - Library imports and versions
- **Training Data** - Loaded from intent_examples.json (225 samples, 16 intents)
- **Text Preprocessing** - Tokenization, stopword removal, lemmatization
- **Feature Extraction** - Bag-of-Words and TF-IDF with visible matrices
- **Model Training** - Naive Bayes classifier trained on all data
- **Evaluation** - Cross-validation metrics
- **Rule Engine** - Pattern-based matching for common queries
- **Hybrid System** - Combining ML and Rules for better accuracy
- **Response Generation** - Templates loaded from response_templates.json
- **End-to-End Demo** - Complete pipeline with outputs at each step

### Architecture Benefits:
1. **Hybrid approach** provides both flexibility (ML) and precision (Rules)
2. **TF-IDF** helps distinguish important words from common words
3. **Rule patterns** ensure reliable handling of common queries
4. **Cross-validation** provides robust evaluation

---
*This notebook aligns with Chapter 3: Architecture, Flowchart, and Evaluation Metrics from your project documentation.*

In [12]:
# Final Summary Statistics
print("=" * 60)
print("FINAL SUMMARY - KEY METRICS")
print("=" * 60)
print()
print("LIBRARY CHATBOT EVALUATION RESULTS")
print("-" * 60)
print(f"Training Samples:     {len(texts)}")
print(f"Number of Intents:    {len(set(labels))}")
print(f"Vocabulary Size:      {len(vocabulary)}")
print("-" * 60)
print(f"CV Accuracy (mean):   {cv_scores.mean():.2%}")
print(f"CV Accuracy (std):    {cv_scores.std():.2%}")
print(f"CV F1-Score (mean):   {cv_f1_scores.mean():.2%}")
print(f"CV F1-Score (std):    {cv_f1_scores.std():.2%}")
print(f"Training Accuracy:    {train_accuracy:.2%}")
print("-" * 60)
print("System Type:           Hybrid (ML + Rules)")
print("ML Model:              Multinomial Naive Bayes")
print("Feature Method:        TF-IDF")
print()
print("Notebook Complete!")


FINAL SUMMARY - KEY METRICS

LIBRARY CHATBOT EVALUATION RESULTS
------------------------------------------------------------
Training Samples:     135
Number of Intents:    16
Vocabulary Size:      100
------------------------------------------------------------
CV Accuracy (mean):   43.70%
CV Accuracy (std):    4.91%
CV F1-Score (mean):   34.91%
CV F1-Score (std):    4.35%
Training Accuracy:    71.11%
------------------------------------------------------------
System Type:           Hybrid (ML + Rules)
ML Model:              Multinomial Naive Bayes
Feature Method:        TF-IDF

Notebook Complete!
